# Sentinel-2 / RadCalNet-OC matchups at the top of atmosphere

Comparison of the Sentinel-2 L1C TOA reflectance with the radcalnet-oc simulation from the AERONET-OC
measurements of the site.

The matchups are computed by `s2_toa_matchup.py` (same folder), from

- the shared AERONET-OC database (`aeronet_oc_db`, `aeronet-oc-db update` to refresh it),
- the hyperspectral Rrs reconstructed with invRrs (`radcalnet_oc.reconstruction`, nadir view),
- the Sentinel-2 L1C subsets around the AERONET-OC sites (`/data/satellite/Sentinel-2/subset/aeronet/L1C`).

Each AERONET-OC measurement within the time window around the overpass is simulated for the sun and
viewing geometry of the scene and band-integrated with the spectral responses of the scene. To update
(only the new scenes are processed):

```
python s2_toa_matchup.py --site AAOT --level lev20 --n-jobs 2
```

`AAOT` is the AERONET-OC site of the Acqua Alta tower since October 2017 (`Venise` before March 2023).

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import scipy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt

plt.rcParams.update({'font.family': 'serif', 'mathtext.fontset': 'stix',
                     'font.size': 16, 'axes.labelsize': 18})
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
opj = os.path.join

In [ ]:
site = 'AAOT'
level = 'lev20'
mdir = '/data/radcalnet/matchup/sentinel2'
figdir = opj('/data/radcalnet/fig/matchup_s2', site)
os.makedirs(figdir, exist_ok=True)

mu = xr.open_dataset(opj(mdir, f'{site}_{level}_s2_toa_matchup.nc')).load()
mu

## Selection of the matchups

- valid pixels (SWIR-1 < `swir_max`, cirrus < `cirrus_max`) on at least `min_frac_valid` of the box;
- spatial homogeneity: coefficient of variation at 865 nm below `max_cv865`;
- at least `min_sim` AERONET-OC measurements within the window;
- thin cirrus: median S2 reflectance of the cirrus band (1375 nm) below `max_cirrus`;
- SWIR residual: S2 - simulation at 1610 nm below `max_dswir` (remaining thin clouds or glint, as in the
  previous version of the matchups).

In [ ]:
min_frac_valid = 0.9
max_cv865 = 0.1
min_sim = 2
max_dswir = 0.007
max_cirrus = 0.002

cirrus = mu.sat_median.sel(wl=1375, drop=True)
cv865 = (mu.sat_std / mu.sat_median).sel(wl=865, drop=True)
dswir = (mu.sat_median - mu.sim_median).sel(wl=1610, drop=True)
good = ((mu.frac_valid >= min_frac_valid) & (cv865 < max_cv865) & (mu.sim_count >= min_sim)
        & (dswir < max_dswir) & (cirrus < max_cirrus))
sel = mu.sel(time=good)
print(f'{mu.sizes["time"]} matchups, {sel.sizes["time"]} selected')
for name, cond in [('frac_valid', mu.frac_valid >= min_frac_valid), ('cv865', cv865 < max_cv865),
                   ('sim_count', mu.sim_count >= min_sim), ('dswir', dswir < max_dswir),
                   ('cirrus', cirrus < max_cirrus)]:
    print(f'  {name}: {int(cond.sum())}')

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(dswir.time, dswir, 'o', ms=4, color='grey', label='all')
ax.plot(dswir.time[good], dswir[good], 'o', ms=4, label='selected')
ax.axhline(max_dswir, color='k', ls='--', lw=1)
ax.set_ylabel(r'$\Delta R_{TOA}(1610)$')
ax.set_title('')
ax.legend()
plt.show()

## Spectra

In [ ]:
n = min(12, sel.sizes['time'])
fig, axs = plt.subplots(3, 4, figsize=(24, 13), sharex=True)
axs = axs.ravel()
for ax, it in zip(axs, np.linspace(0, sel.sizes['time'] - 1, n).astype(int)):
    d = sel.isel(time=it)
    recs = mu.rec_Rtoa_spectrum.where(mu.rec_scene == d.time, drop=True)
    ax.plot(mu.wl_spectrum, recs.values.T, color='orange', lw=0.5, alpha=0.5)
    ax.plot(mu.wl_spectrum, d.sim_spectrum_median, color='orange', lw=1.2, label='RadCalNet-OC')
    ax.errorbar(d.wl_true, d.sat_median, yerr=[d.sat_median - d.sat_q25, d.sat_q75 - d.sat_median],
                fmt='o', color='b', ms=5, capsize=3, label='Sentinel-2')
    ax.plot(d.wl_true, d.sim_median, 's', mfc='none', color='r', ms=7, label='RadCalNet-OC band')
    ax.set_title(str(d.time.dt.strftime('%Y-%m-%d').values) + r', SZA=%.0f, VZA=%.0f, AOT=%.2f' %
                 (d.sza, d.vza, d.aot550), fontsize=14)
    ax.set_xlim(400, 2300)
    ax.minorticks_on()
axs[0].legend(fontsize=12)
for ax in axs[8:]:
    ax.set_xlabel(r'$Wavelength\ (nm)$')
for ax in axs[::4]:
    ax.set_ylabel(r'$R_{TOA}\ (-)$')
plt.tight_layout()
plt.savefig(opj(figdir, f'{site}_{level}_spectra.png'), dpi=150)
plt.show()

## Time series

In [ ]:
bands = [443, 490, 560, 665, 705, 783, 865, 1610]
fig, axs = plt.subplots(len(bands), 1, figsize=(20, 3.2 * len(bands)), sharex=True)
for ax, wl in zip(axs, bands):
    d = sel.sel(wl=wl)
    ax.errorbar(d.time.values, d.sat_median, yerr=[d.sat_median - d.sat_q25, d.sat_q75 - d.sat_median],
                fmt='o', ms=4, color='b', alpha=0.7, capsize=2, label='Sentinel-2')
    ax.errorbar(d.time.values, d.sim_median, yerr=[d.sim_median - d.sim_q25, d.sim_q75 - d.sim_median],
                fmt='s', ms=4, color='orange', alpha=0.7, capsize=2, label='RadCalNet-OC')
    ax.set_ylabel(r'$R_{TOA}(%i)$' % wl)
    ax.minorticks_on()
axs[0].legend(ncol=2)
plt.tight_layout()
plt.savefig(opj(figdir, f'{site}_{level}_timeseries.png'), dpi=150)
plt.show()

## Scatter plots

Statistics: linear regression, RMSE, bias, median symmetric accuracy $\epsilon$ and median bias $\beta$
(Pahlevan et al., 2020).

In [ ]:
def add_stats(x, y, ax, fontsize=12):
    regr = sp.stats.linregress(x, y)
    rmse = np.sqrt(np.mean((y - x) ** 2))
    bias = np.mean(y - x)
    eps = 10 ** np.median(np.abs(np.log10(y / x))) - 1
    beta = np.sign(np.median(np.log10(y / x))) * (10 ** np.abs(np.median(np.log10(y / x))) - 1)
    stats = (r'$y = %.3f x %+.4f$' % (regr.slope, regr.intercept) +
             '\n$r = %.3f$' % regr.rvalue + '\n$rmse = %.4f$' % rmse + '\n$bias = %.4f$' % bias +
             '\n$\\epsilon = %.1f\\ \\%%$' % (100 * eps) + '\n$\\beta = %+.1f\\ \\%%$' % (100 * beta) +
             '\n$N = %i$' % len(x))
    ax.axline((0, regr.intercept), slope=regr.slope, ls='--', lw=1.5, c='gray')
    ax.axline((0, 0), slope=1, lw=1, c='k')
    ax.text(0.98, 0.02, stats, fontsize=fontsize, va='bottom', ha='right', transform=ax.transAxes)
    return dict(slope=regr.slope, intercept=regr.intercept, r=regr.rvalue, rmse=rmse, bias=bias,
                eps=eps, beta=beta, N=len(x))


cmap = plt.cm.Spectral_r
norm = mpl.colors.Normalize(vmin=440, vmax=870)
wls = [wl for wl in sel.wl.values if wl not in (945, 1375)]
fig, axs = plt.subplots(3, 4, figsize=(22, 16))
axs = axs.ravel()
table = {}
for ax, wl in zip(axs, wls):
    d = sel.sel(wl=wl)
    ok = d.sat_median.notnull() & d.sim_median.notnull()
    x, y = d.sim_median.values[ok], d.sat_median.values[ok]
    ax.errorbar(x, y, xerr=d.sim_std.values[ok], yerr=d.sat_std.values[ok], fmt='none', color='gray',
                lw=0.6, capsize=2, zorder=1)
    ax.plot(x, y, 'o', color=cmap(norm(min(wl, 870))), ms=4, alpha=0.8, zorder=2)
    table[wl] = add_stats(x, y, ax)
    lim = [0, 1.05 * max(x.max(), y.max())]
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_title(f'{wl} nm')
    ax.set_xlabel(r'$RadCalNet\text{-}OC$')
    ax.set_ylabel(r'$Sentinel\text{-}2$')
    ax.minorticks_on()
for ax in axs[len(wls):]:
    ax.set_axis_off()
plt.tight_layout()
plt.savefig(opj(figdir, f'{site}_{level}_scatter_per_band.png'), dpi=150)
plt.show()
pd.DataFrame(table).T

## Residual: wind speed and aerosols

Difference S2 − RadCalNet-OC of the selected matchups (clear pixels) far from the specular direction
(glint angle > `min_glint_angle`), as a function of the wind speed and of the AOT of the AERONET-OC
measurements. Grey line: upper bound of the whitecap reflectance (Monahan and O'Muircheartaigh, 1980,
coverage x 0.22), not in the simulation. See the notebook `notebook/sentinel-2/sentinel2_radcalnet_oc_toa_sunglint.ipynb`
for the comparison with the sunglint.

In [ ]:
min_glint_angle = 40
far = sel.sel(time=sel.glint_angle > min_glint_angle)
res = far.sat_median - far.sim_median
print(f'{far.sizes["time"]} selected scenes with a glint angle > {min_glint_angle} deg')
u = np.linspace(0, 12, 100)
whitecap = 0.22 * 2.95e-6 * u ** 3.52
wl_check = [560, 865, 1610, 2190]
fig, axs = plt.subplots(2, len(wl_check), figsize=(6 * len(wl_check), 10), sharey='col')
for j, wl in enumerate(wl_check):
    y = res.sel(wl=wl).values
    for i, (var, xlabel) in enumerate([('wind', r'wind speed (m s$^{-1}$)'), ('aot550', 'AOT(550)')]):
        ax = axs[i, j]
        x = far[var].values
        ok = np.isfinite(x) & np.isfinite(y)
        ax.plot(x[ok], y[ok], 'o', ms=5, alpha=0.7)
        if ok.sum() > 2:
            regr = sp.stats.linregress(x[ok], y[ok])
            ax.axline((0, regr.intercept), slope=regr.slope, color='tab:red', ls='--', lw=1)
            ax.text(0.02, 0.98, 'r = %.2f\nslope = %.2e\nintercept = %.4f\nN = %i' %
                    (regr.rvalue, regr.slope, regr.intercept, ok.sum()), transform=ax.transAxes,
                    va='top', fontsize=12)
        if var == 'wind':
            ax.plot(u, whitecap, color='grey', lw=1.5)
        ax.axhline(0, color='k', lw=1)
        ax.set_xlabel(xlabel)
        ax.minorticks_on()
    axs[0, j].set_title(f'{wl} nm')
    axs[0, j].set_ylim(-0.005, 0.01)
for ax in axs[:, 0]:
    ax.set_ylabel('S2 - RadCalNet-OC')
plt.tight_layout()
plt.savefig(opj(figdir, f'{site}_{level}_residual_wind_aot.png'), dpi=150)
plt.show()

## Ratio Sentinel-2 / RadCalNet-OC per band and per satellite

Median ratio and interquartile range (vicarious gain estimate).

In [ ]:
ratio = (sel.sat_median / sel.sim_median)
sat = xr.DataArray([str(p)[:3] for p in sel['product'].values], dims='time')
fig, ax = plt.subplots(figsize=(12, 6))
for k, (name, r) in enumerate(ratio.groupby(sat)):
    r = r.sel(wl=wls)
    med = r.median('time')
    ax.errorbar(np.arange(len(wls)) + 0.15 * (k - 1), med,
                yerr=[med - r.quantile(0.25, 'time'), r.quantile(0.75, 'time') - med],
                fmt='o', capsize=3, label=f'{name} (N = {r.sizes["time"]})')
ax.axhline(1, color='k', lw=1)
ax.set_xticks(np.arange(len(wls)), [str(w) for w in wls])
ax.set_xlabel(r'$Band\ (nm)$')
ax.set_ylabel(r'$R_{TOA}^{S2} / R_{TOA}^{RadCalNet-OC}$')
ax.set_ylim(0.85, 1.25)
ax.legend()
ax.minorticks_on()
plt.tight_layout()
plt.savefig(opj(figdir, f'{site}_{level}_ratio.png'), dpi=150)
plt.show()
ratio.sel(wl=wls).to_dataframe('ratio').groupby('wl').ratio.describe()